In [ ]:
import boto3
import pandas as pd
from io import StringIO

# --------------------------------------------------
# 1. Connect to S3
# --------------------------------------------------

s3 = boto3.resource("s3")

bucket_name = "my-company-bucket"
folder_prefix = "sales/invoices/"

bucket = s3.Bucket(bucket_name)

# --------------------------------------------------
# 2. List objects and identify CSV files
# --------------------------------------------------

csv_files = []

for obj in bucket.objects.filter(Prefix=folder_prefix):

    key = obj.key

    # Find the position of ".csv" in the object key
    if key.find(".csv") != -1:

        # Split the S3 key using "/" to get folder/file parts
        path_parts = key.split("/")

        # Last part is the file name
        file_name = path_parts[-1]
        if '.csv' in file_name:
           print("CSV file found:", file_name)
           print("Full S3 key:", key)
           csv_files.append(key)

print("Total CSV files:", len(csv_files))

# --------------------------------------------------
# 3. Read all CSV files into DataFrames
# --------------------------------------------------

dataframes = []

for key in csv_files:

    response = s3.meta.client.get_object(
        Bucket=bucket_name,
        Key=key
    )

    csv_content = response["Body"].read().decode("utf-8")

    df = pd.read_csv(StringIO(csv_content))

    dataframes.append(df)

# Combine all files into one DataFrame
if dataframes:

    combined_df = pd.concat(dataframes, ignore_index=True)

    print("Combined DataFrame:")
    print(combined_df.head())

else:
    print("No CSV files found.")

# --------------------------------------------------
# 4. Clean and prepare data
# --------------------------------------------------

# Convert numeric columns
combined_df["Quantity"] = pd.to_numeric(
    combined_df["Quantity"], errors="coerce"
)

combined_df["Price"] = pd.to_numeric(
    combined_df["Price"], errors="coerce"
)

# Remove rows where Quantity or Price is missing
combined_df = combined_df.dropna(
    subset=["Quantity", "Price"]
)

# Calculate sales revenue
combined_df["Sales_Revenue"] = (
    combined_df["Quantity"] * combined_df["Price"]
)

# --------------------------------------------------
# 5. Aggregate invoice data
# --------------------------------------------------

group_columns = [
    "Product",
    "Region",
    "Customer",
    "State",
    "Product_Category"
]

aggregated_df = (
    combined_df
    .groupby(group_columns, as_index=False)
    .agg(
        Total_Quantity=("Quantity", "sum"),
        Total_Revenue=("Sales_Revenue", "sum"),
        Average_Unit_Price=("Price", "mean")
    )
)

print("Aggregated DataFrame:")
print(aggregated_df.head())

# --------------------------------------------------
# 6. Save aggregated data as a new CSV in S3
# --------------------------------------------------

output_key = "sales/aggregated/invoice_data_aggregated.csv"

csv_buffer = StringIO()
aggregated_df.to_csv(csv_buffer, index=False)

s3.meta.client.put_object(
    Bucket=bucket_name,
    Key=output_key,
    Body=csv_buffer.getvalue(),
    ContentType="text/csv"
)

print("Aggregated file uploaded to:")
print(f"s3://{bucket_name}/{output_key}")